In [1]:
import os
import pandas as pd
from pandas import json_normalize
from yaml import safe_load
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objs as go

In [2]:
# print the whole dataframe
def print_df(df):
    styled_df = df.style.set_table_styles([
        {'selector': 'tr:hover', 'props': [('background-color', 'yellow')]}
        ])
    return styled_df

In [3]:
def print_statistics(dfs_list, dfs_name, add_columns=None):
    for df, name in zip(dfs_list, dfs_name):

        selected_columns = ['system_complexity', 'energy_training', 'energy_test']
        if add_columns:
            selected_columns.append(add_columns)
        statistics = df[selected_columns].describe()

        # Rename the '50%' row to 'median'
        statistics.rename(index={'50%': 'median'}, inplace=True)

        print(f"Statistics for df {name} in terms of the metric considered")
        formatted_statistics = statistics.applymap(lambda x: f'{x:.2f}')
        print(formatted_statistics)
        print('\n')

In [4]:
def round_to(value, decimal_places):
    return value.round(decimal_places)

In [5]:
## convert string to an int number
def convert_suffix_to_int(number_str):
    suffixes = {'K': 10**3, 'M': 10**6}
    if isinstance(number_str, str) and (number_str[-1] in suffixes or number_str[-1] == 'B'):
        number_str = number_str.upper()
        if number_str[-1] == 'B':
            multiplier = suffixes['M']
            value = int(float(number_str[:-2]) * multiplier)
        else:
            multiplier = suffixes[number_str[-1]]
            value = int(float(number_str[:-1]) * multiplier)
        return value
    else:
        return int(number_str)



In [6]:
# Calculate statistics for selected columns, additiional columns can be added
def print_statistics(dfs_list, dfs_name, selected_columns):
    for df, name in zip(dfs_list, dfs_name):

        statistics = df[selected_columns].describe()

        # Rename the '50%' row to 'median'
        statistics.rename(index={'50%': 'median'}, inplace=True)

        # Print the statisti
        print(f"Statistics for df {name} in terms of the metric considered")
        formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')
        print(formatted_statistics)
        print('\n')

In [7]:
def scatter_2024(
    df1,
    x_lab,
    y_lab,
    x_axes_lab,
    y_axes_lab,
    code,
    title,
    type_xaxes='linear',
    type_yaxes='linear',
    marker_size=10,
    save=True,
    legend=True,
):
    colors = ['green', 'orange']

    # Extracting baseline data
    baseline_24 = df1[df1['code'] == 'Baseline 2024']


    # Initialize figure with subplots
    fig = make_subplots(
        rows=1, cols=1,  # Only 1 subplot
        column_widths=[0.5],
        row_heights=[0.3],
        specs=[[{"type": "scatter"}]]
    )

    # Adding the 2024 data
    fig.add_trace(
        go.Scatter(
            x=df1[x_lab],
            y=df1[y_lab],
            text=df1[code],
            mode='markers',
            name='Entries 2024',
            marker=dict(size=marker_size, color='black')
        ),
        row=1, col=1
    )


    # Adding baselines
    for index, (base, label) in enumerate(zip([baseline_24],
                                              ['Baseline 2024'])):
        fig.add_trace(
            go.Scatter(
                x=base[x_lab],
                y=base[y_lab],
                text=base[code],
                mode='markers',
                name=label,
                marker=dict(size=marker_size, color=colors[index])
            ),
            row=1, col=1
        )

    # Layout and axis configuration
    # fig.update_layout(title=title, title_x=0.5)  # Center title

    # Add x-label and y-label to the first subplot
    fig.update_xaxes(title_text=x_axes_lab, type=type_xaxes, row=1, col=1)
    fig.update_yaxes(title_text=y_axes_lab, type=type_yaxes, row=1, col=1)

    fig.update_layout(width=280,
                height=240,
                showlegend=legend,
                margin=dict(l=0, r=0, t=0, b=0),  # Adjust top margin to make space for legend
                legend=dict(
                    orientation="h",
                    yanchor="bottom",
                    y=1.02,
                    xanchor="right",
                    x=1))

    if save:
        # fig.write_image(title)
        pass

    fig.show()



In [8]:
def scatter_bothyears(
    df1,
    df2,
    x_lab,
    y_lab,
    x_axes_lab,
    y_axes_lab,
    code,
    title,
    type_xaxes='linear',
    type_yaxes='linear',
    x_range=[None, None],
    y_range=[None,None],
    marker_size=8,
    top_10=False,
    save=True,
    legend=True,
):
    colors = ['green', 'orange']

    # Extracting baseline data
    baseline_24 = df1[df1['code'] == 'Baseline 2024']
    baseline_23_1 = df2[df2['code'] == 'Baseline_task4a_1']
    baseline_23_2 = df2[df2['code'] == 'Baseline_task4a_2']


    if top_10 == True:
        # Removing baseline data from the DataFrames
        df1 = df1[df1['code'] != 'Baseline 2024']
        df2 = df2[(df2['code'] != 'Baseline_task4a_1') & (df2['code'] != 'Baseline_task4a_2')]

        df1 = df1.dropna(subset=[x_lab, y_lab])
        df2 = df2.dropna(subset=[x_lab, y_lab])

        df1.drop_duplicates(subset=[x_lab,y_lab], inplace=True)
        df2.drop_duplicates(subset=[x_lab,y_lab], inplace=True)

        df1 = df1.sort_values(by=x_lab, ascending=False)[:10]
        df2 = df2.sort_values(by=x_lab, ascending=False)[:10]

        df1.reset_index(inplace=True, drop=True)
        df2.reset_index(inplace=True, drop=True)

    # Initialize figure with subplots
    fig = make_subplots(
        rows=1, cols=1,  # Only 1 subplot
        row_heights=[0.3],
        specs=[[{"type": "scatter"}]]
    )




    # Adding the 2023 data
    fig.add_trace(
        go.Scatter(
            x=df2[x_lab],
            y=df2[y_lab],
            text=df2[code],
            mode='markers',
            name='Entries 2023',
            marker=dict(size=marker_size, color='blue')
        ),
        row=1, col=1
    )

    # Adding the 2024 data
    fig.add_trace(
        go.Scatter(
            x=df1[x_lab],
            y=df1[y_lab],
            text=df1[code],
            mode='markers',
            name='Entries 2024',
            marker=dict(size=marker_size, color='black')
        ),
        row=1, col=1
    )

            # Adding baselines
    for index, (base, label) in enumerate(zip([baseline_24, baseline_23_2],
                                              ['Baseline 2024', 'Baseline 2023'])):
        fig.add_trace(
            go.Scatter(
                x=base[x_lab],
                y=base[y_lab],
                text=base[code],
                mode='markers',
                name=label,
                marker=dict(size=marker_size, color=colors[index])
            ),
            row=1, col=1
        )

    # Layout and axis configuration
    # fig.update_layout(title=title, title_x=0.5)  # Center title
    fig.update_layout(width=400,
                    height=200,
                    showlegend=legend,
                    margin=dict(l=0, r=0, t=0, b=0),  # Adjust top margin to make space for legend
                    legend=dict(
                        yanchor="top",
                        y=0.99,
                        xanchor="right",
                        x=0.99
                            ))

    # Add x-label and y-label to the first subplot
    fig.update_xaxes(title_text=x_axes_lab, type=type_xaxes, range=x_range, row=1, col=1)
    fig.update_yaxes(title_text=y_axes_lab, type=type_yaxes, range=y_range, row=1, col=1)
    if save:
        # fig.write_image(title)
        pass

    fig.show()


In [9]:
def scatter_2024_ensemble_four(
    df_e,
    df_ne,
    x_lab,
    y_lab,
    x_axes_lab,
    y_axes_lab,
    code,
    title,
    type_xaxes='linear',
    type_yaxes='linear',
    marker_size=8,
    x_range=[None, None],
    y_range=[None, None],
    save=True,
    legend=True,
):
    color_baseline = ['green']

    # Extracting baseline data
    baseline_24 = df_ne[df_ne['code'] == 'Baseline 2024']
    # Drop NaN values before calculating the min and max
    df_e = df_e.dropna(subset=[x_lab, y_lab])
    df_ne = df_ne.dropna(subset=[x_lab, y_lab])

    # Initialize figure with subplots
    fig = make_subplots(
        rows=1, cols=1,  # Only 1 subplot
        column_widths=[0.5],
        row_heights=[0.3],
        specs=[[{"type": "scatter"}]]
    )

    # Define the baseline values for comparison
    baseline_performance = baseline_24[x_lab].values[0]
    baseline_energy = baseline_24[y_lab].values[0]


    fig.add_trace(
        go.Scatter(
            x=df_e[x_lab],
            y=df_e[y_lab],
            text=df_e['code'],
            mode='markers',
            name='Ensemble',
            marker=dict(size=marker_size, color='#af64f6')
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df_ne[x_lab],
            y=df_ne[y_lab],
            text=df_ne['code'],
            mode='markers',
            name='Non-Ensemble',
            marker=dict(size=marker_size, color='#f95543')
        ),
        row=1, col=1
    )

    # Get the limits of the axes from the cleaned data to ensure the lines stay within bounds
    x_min, x_max = min(df_e[x_lab].min(), df_ne[x_lab].min()), max(df_e[x_lab].max(), df_ne[x_lab].max())
    y_min, y_max = min(df_e[y_lab].min(), df_ne[y_lab].min()), max(df_e[y_lab].max(), df_ne[y_lab].max())

    if type_yaxes == 'log':
        y_min = max(y_min, 4)  # Set a small positive value as the lower bound

    # Draw horizontal dotted line for baseline performance
    fig.add_shape(
        type='line',
        x0=x_range[0], x1=x_range[1],
        y0=baseline_performance, y1=baseline_performance,
        line=dict(color='green', width=2, dash='dot'),
        row=1, col=1
    )

    # Draw vertical dotted line for baseline energy
    fig.add_shape(
        type='line',
        x0=baseline_energy, x1=baseline_energy,
        y0=y_range[0], y1=y_range[1],
        line=dict(color='green', width=2, dash='dot'),
        row=1, col=1
    )

    # Add trace for Baseline 2024
    fig.add_trace(
        go.Scatter(
            x=baseline_24[x_lab],
            y=baseline_24[y_lab],
            text=baseline_24[code],
            mode='markers',
            name='Baseline 2024',
            marker=dict(size=marker_size, color=color_baseline)
        ),
        row=1, col=1
    )

    # Add x-label and y-label to the first subplot
    fig.update_xaxes(title_text=x_axes_lab, type=type_xaxes, row=1, col=1)
    fig.update_yaxes(title_text=y_axes_lab, type=type_yaxes, row=1, col=1)

    fig.update_layout(width=400,
                height=200,
                showlegend=legend,
                margin=dict(l=0, r=0, t=0, b=0),  # Adjust top margin to make space for legend
                legend=dict(
                    yanchor="top",
                    y=0.99,
                    xanchor="left",
                    x=0.01
                        ))

    if save:
        # fig.write_image(title)
        pass

    fig.show()



# 2022 ENTRIES

In [10]:
from google.colab import drive
drive.mount('/content/drive')

os.chdir('drive/My Drive/Post_Doc_INRIA/ICASSP_25')


Mounted at /content/drive


In [11]:
# read the data from file results_task4_entries, retrieving all the informations for 2022 systems
fname_22 = 'results_dcase_2022.yaml'

with open(fname_22, 'r') as f:
    df_all_22 = json_normalize(safe_load(f))


In [12]:
#gather only the columns we are going to need
columns_list = ["energy_training", "energy_test", "system_complexity", "code", "system_ensemble_method_subsystem_count", "PSDS_1_all", "PSDS_2_all"]
df_e_22 = df_all_22[columns_list]

In [13]:
# manually processing the system complexity of some submissions because of different formats in the data
df_e_22.loc[df_e_22['system_complexity'] == '1.7MB', 'system_complexity'] = '1.7M'
df_e_22.loc[df_e_22['system_complexity'] == '1.1MB', 'system_complexity'] = '1.1M'
df_e_22.loc[df_e_22['system_complexity'] == '4.2MB', 'system_complexity'] = '4.2M'
df_e_22.loc[df_e_22['system_complexity'] == 'Trainable 1.7 M non-Trainable 1.7M', 'system_complexity'] = '3.4M'
df_e_22.loc[df_e_22['system_complexity'] == 'Trainable 1.1 M non-Trainable 1.1M', 'system_complexity'] = '2.2M'

In [14]:
df_e_22.reset_index(inplace=True, drop=True)
subset = ['system_complexity', 'energy_training', 'energy_test']
df_e_22.dropna(subset=subset, inplace=True) #drop null values
df_e_22[subset[0]] = df_e_22[subset[0]].apply(convert_suffix_to_int) # convert system complexity to int values
df_e_22.drop_duplicates(subset=subset, inplace=True) # drop networks that have been repeated more than once
df_e_22.reset_index(inplace=True, drop=True)

<ipython-input-14-9e2ca4d35411>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_e_22.dropna(subset=subset, inplace=True) #drop null values
<ipython-input-14-9e2ca4d35411>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_e_22[subset[0]] = df_e_22[subset[0]].apply(convert_suffix_to_int) # convert system complexity to int values
<ipython-input-14-9e2ca4d35411>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html

# 2023 ENTRIES

In [15]:
# read the data from file results_task4_entries_energy
fname_23 = 'results_dcase_2023.yaml'

with open(fname_23, 'r') as f:
    df_all_23 = json_normalize(safe_load(f))

# psds1 to only one value
for label in ['psds1_eval_full', 'psds2_eval_full']:
    df_all_23[label] = df_all_23[label].str.extract(r'(\d+\.\d+)')
    df_all_23[label] = df_all_23[label].astype(float)

In [16]:
columns_list = ['energy_training',
                "energy_test",
                'energy_baseline',
                'energy_training_normalized',
                'energy_test_normalized',
                'ew_psds1_eval_full_train',
                'ew_psds1_eval_full_test',
                'ew_psds2_eval_full_train',
                'ew_psds2_eval_full_test',
                'system_complexity_params',
                'system_complexity_time',
                'system_name',
                'system_classifier',
                'system_ensemble_method_subsystem_count',
                'psds1_eval_full',
                'psds2_eval_full',
                'macs']

df_e_23 = df_all_23[columns_list]

# we rename the columns so we can use the same name reference for both 2022 an 2023 dataframe
df_e_23.rename(columns = {'system_name':'code'}, inplace = True)
df_e_23.rename(columns = {'system_complexity_params':'system_complexity'}, inplace = True)
df_e_23.rename(columns = {'psds1_eval_full':'PSDS_1_all'}, inplace = True)
df_e_23.rename(columns = {'psds2_eval_full':'PSDS_2_all'}, inplace = True)
df_e_23.reset_index(inplace=True, drop=True)

<ipython-input-16-8adc7d4e364c>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_e_23.rename(columns = {'system_name':'code'}, inplace = True)
<ipython-input-16-8adc7d4e364c>:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_e_23.rename(columns = {'system_complexity_params':'system_complexity'}, inplace = True)
<ipython-input-16-8adc7d4e364c>:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_e_23.rename(columns = {'psds1_eval

In [17]:
df_e_23.drop_duplicates(subset=subset, inplace=True)
df_e_23.reset_index(inplace=True, drop=True)


<ipython-input-17-a7937e8defc9>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_e_23.drop_duplicates(subset=subset, inplace=True)


# 2024 ENTRIES

In [18]:
fname_24 = 'results_dcase_2024.yaml'

with open(fname_24, 'r') as f:
    df_all_24 = json_normalize(safe_load(f))

for label in ['psds1_desed_eval_bootstrap', 'mean_sbpauc_maestro_eval_fullset']:
    df_all_24[label] = df_all_24[label].str.extract(r'(\d+\.\d+)')
    df_all_24[label] = df_all_24[label].astype(float)

df_all_24['ew_total_training_ranking'] = df_all_24['ew_total_training_psds1_desed_eval_bootstrap'] + df_all_24['ew_total_training_mean_sbpauc_maestro_eval_fullset']

#Correction on how energy test is normalized to fit 2023 methodology
df_all_24['energy_consumption_norm_test'] = df_all_24['energy_consumption_submission_test'] * (0.0299 / df_all_24['energy_consumption_baseline_10epochs'])
df_all_24['gpu_energy_norm_test'] = df_all_24['gpu_energy_submission_test'] * ( 0.0103 / df_all_24['gpu_energy_baseline_10epochs'])

In [19]:
columns_list = ['energy_consumption_submission_training',
                'energy_consumption_submission_test',
                'energy_consumption_baseline_10epochs',
                'energy_consumption_baseline_test',
                'gpu_energy_submission_training',
                'gpu_energy_submission_test',
                'gpu_energy_baseline_10epochs',
                'gpu_energy_baseline_test',
                'energy_consumption_norm_training',
                'energy_consumption_norm_test',
                'gpu_energy_norm_training',
                'gpu_energy_norm_test',
                'psds1_desed_eval_bootstrap',
                'mean_sbpauc_maestro_eval_fullset',
                'ranking_score',
                'system_name',
                'system_classifier',
                'system_complexity_params',
                'system_complexity_time',
                'system_ensemble_method_subsystem_count',
                'macs',
                'ew_total_training_ranking',
                'ew_total_training_psds1_desed_eval_bootstrap',
                'ew_total_training_mean_sbpauc_maestro_eval_fullset']

df_e_24 = df_all_24[columns_list]

df_e_24.rename(columns = {'system_name':'code'}, inplace = True)
df_e_24.rename(columns = {'system_complexity_params':'system_complexity'}, inplace = True)

df_e_24.rename(columns = {'energy_consumption_submission_training':'energy_training'}, inplace = True)
df_e_24.rename(columns = {'energy_consumption_submission_test':'energy_test'}, inplace = True)
df_e_24.rename(columns = {'energy_consumption_norm_training':'energy_training_normalized'}, inplace = True)
df_e_24.rename(columns = {'energy_consumption_norm_test':'energy_test_normalized'}, inplace = True)
df_e_24.rename(columns= {'energy_consumption_baseline_10epochs':'energy_baseline'}, inplace = True)

df_e_24.rename(columns = {'psds1_desed_eval_bootstrap':'PSDS_1_all'}, inplace = True)
df_e_24.rename(columns = {'mean_sbpauc_maestro_eval_fullset':'PAUC_all'}, inplace = True)

df_e_24.rename(columns = {'ew_total_training_ranking':'ew_ranking'}, inplace = True)
df_e_24.rename(columns = {'ew_total_training_psds1_desed_eval_bootstrap':'ew_psds'}, inplace = True)
df_e_24.rename(columns = {'ew_total_training_mean_sbpauc_maestro_eval_fullset':'ew_pauc'}, inplace = True)


<ipython-input-19-40e9faf8e5e0>:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_e_24.rename(columns = {'system_name':'code'}, inplace = True)
<ipython-input-19-40e9faf8e5e0>:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_e_24.rename(columns = {'system_complexity_params':'system_complexity'}, inplace = True)
<ipython-input-19-40e9faf8e5e0>:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_e_24.rename(columns = {'energy_con

In [20]:
drop_systems = ['Zhang_BUPT_task4_2', 'Cai_USTC_task4_1', 'Cai_USTC_task4_2', 'Cai_USTC_task4_3', 'Cai_USTC_task4_4']

df_e_24 = df_e_24[~df_e_24['code'].isin(drop_systems)]
df_e_24.drop_duplicates(subset=subset, inplace=True)
df_e_24.reset_index(inplace=True, drop=True)

<ipython-input-20-dd0d12eed148>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_e_24.drop_duplicates(subset=subset, inplace=True)


In [21]:
df_e_24.loc[df_e_24['code'] == 'Cornell_CMU_task4_1', 'code'] = "Baseline 2024"

In [22]:
print(f"All data : \n")
print(f"DCASE 2022 entries: {len(df_all_22)}\n")
print(f"DCASE 2023 entries: {len(df_all_23)}\n")
print(f"DCASE 2024 entries: {len(df_all_24)}\n")

All data : 

DCASE 2022 entries: 101

DCASE 2023 entries: 84

DCASE 2024 entries: 42



In [ ]:
print(f"After the pre-process of the data we are able to analysize: \n")
print(f"DCASE 2022 entries: {len(df_e_22)}\n")
print(f"DCASE 2023 entries: {len(df_e_23)}\n")
print(f"DCASE 2024 entries: {len(df_e_24)}\n")


After the pre-process of the data we are able to analysize: 

DCASE 2022 entries: 60

DCASE 2023 entries: 64

DCASE 2024 entries: 35



## General comparisons (2022 - 2024)

In [ ]:
print_statistics([df_e_22, df_e_23, df_e_24], ['DCASE_22', 'DCASE23','DCASE_24'],['energy_training', 'energy_test'])

Statistics for df DCASE_22 in terms of the metric considered
       energy_training energy_test
count           60.000      60.000
mean            12.960       0.121
std             22.642       0.297
min              0.733       0.002
25%              1.815       0.010
median           3.699       0.026
75%             17.291       0.046
max            151.415       1.772


Statistics for df DCASE23 in terms of the metric considered
       energy_training energy_test
count           64.000      64.000
mean            28.109       0.211
std             79.355       0.366
min              0.036       0.001
25%              1.617       0.019
median           4.295       0.035
75%             13.975       0.283
max            446.600       1.905


Statistics for df DCASE_24 in terms of the metric considered
       energy_training energy_test
count           35.000      35.000
mean            25.067       0.296
std             42.459       0.336
min              0.155       0.011
25%      

/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')
/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')
/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')


In [ ]:
print_statistics([df_e_23, df_e_24], ['DCASE23','DCASE_24'], ['macs'])

Statistics for df DCASE23 in terms of the metric considered
                      macs
count               64.000
mean      645156947859.375
std      2917896202735.121
min          105200000.000
25%         3497000000.000
median      9741000000.000
75%       123926750000.000
max     21036000000000.000


Statistics for df DCASE_24 in terms of the metric considered
                    macs
count             35.000
mean    120931912394.971
std     214614204141.854
min        345260000.000
25%       1740000000.000
median   20822000000.000
75%     107969000000.000
max     934958075904.000




/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')
/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')


## Energy normalized comparisons (2023 - 2024)

In [ ]:
df_e_23 = df_e_23[~df_e_23['code'].str.startswith('Chen_CHT')] #energy baseline is not well reported so the normalization is wrong
df_e_23.reset_index(inplace=True, drop=True)

In [ ]:
print_statistics([df_e_23, df_e_24], ['DCASE23','DCASE_24'], ['energy_baseline'])

Statistics for df DCASE23 in terms of the metric considered
       energy_baseline
count           60.000
mean             0.030
std              0.020
min              0.006
25%              0.015
median           0.027
75%              0.045
max              0.075


Statistics for df DCASE_24 in terms of the metric considered
       energy_baseline
count           30.000
mean             0.052
std              0.023
min              0.013
25%              0.038
median           0.048
75%              0.073
max              0.088




/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')
/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')


In [ ]:
print_statistics([df_e_23, df_e_24], ['DCASE23','DCASE_24'], ['macs', 'energy_training_normalized', 'energy_test_normalized'])

Statistics for df DCASE23 in terms of the metric considered
                      macs energy_training_normalized energy_test_normalized
count               60.000                     60.000                 60.000
mean      685925761050.000                     20.159                  0.336
std      3010685343803.701                     39.933                  0.873
min          105200000.000                      0.192                  0.002
25%         3078750000.000                      2.506                  0.019
median      9741000000.000                      6.585                  0.053
75%       149190500000.000                     13.144                  0.367
max     21036000000000.000                    190.549                  6.240


Statistics for df DCASE_24 in terms of the metric considered
                    macs energy_training_normalized energy_test_normalized
count             35.000                     30.000                 30.000
mean    120931912394.971          

/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')
/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')


In [ ]:
print_statistics([df_e_23, df_e_24], ['DCASE23','DCASE_24'], ['macs', 'energy_training_normalized', 'energy_test_normalized'])

Statistics for df DCASE23 in terms of the metric considered
                      macs energy_training_normalized energy_test_normalized
count               60.000                     60.000                 60.000
mean      685925761050.000                     20.159                  0.336
std      3010685343803.701                     39.933                  0.873
min          105200000.000                      0.192                  0.002
25%         3078750000.000                      2.506                  0.019
median      9741000000.000                      6.585                  0.053
75%       149190500000.000                     13.144                  0.367
max     21036000000000.000                    190.549                  6.240


Statistics for df DCASE_24 in terms of the metric considered
                    macs energy_training_normalized energy_test_normalized
count             35.000                     30.000                 30.000
mean    120931912394.971          

/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')
/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')


First conclusions on the median
- nb parameters x2
- macs x2
- energy_train -4 kWh
- energy_norm_train -4kWh
- energy_test +0.10 kWh
- energy_norm_test +0.10 kWh

## GPU energy consumption (2024)

In [ ]:
df_e_24['pourc_gpu_training'] = df_e_24['gpu_energy_submission_training']/df_e_24['energy_training'] * 100
df_e_24['pourc_gpu_test'] = df_e_24['gpu_energy_submission_test']/df_e_24['energy_test'] * 100
df_e_24['pourc_gpu_training_baseline'] = df_e_24['gpu_energy_baseline_10epochs']/df_e_24['energy_baseline'] * 100

In [ ]:
df_e_24[['pourc_gpu_training', 'pourc_gpu_test']].describe()

,pourc_gpu_training,pourc_gpu_test
count,31.000000,31.000000
mean,52.697134,38.001514
std,16.983878,21.420208
min,27.145866,5.000000
25%,42.705691,22.571864
50%,47.821229,39.207048
75%,61.073201,45.288248
max,87.063738,85.435978


In [ ]:
print_statistics([df_e_23, df_e_24], ['DCASE23','DCASE_24'], ['energy_baseline'])

Statistics for df DCASE23 in terms of the metric considered
       energy_baseline
count           60.000
mean             0.030
std              0.020
min              0.006
25%              0.015
median           0.027
75%              0.045
max              0.075


Statistics for df DCASE_24 in terms of the metric considered
       energy_baseline
count           30.000
mean             0.052
std              0.023
min              0.013
25%              0.038
median           0.048
75%              0.073
max              0.088




/tmp/ipykernel_10801/3221466737.py:12: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

/tmp/ipykernel_10801/3221466737.py:12: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



In [ ]:
#No-ensemble
df_22_noensemble = df_e_22.loc[df_e_22['system_ensemble_method_subsystem_count'].isna()]
df_23_noensemble = df_e_23.loc[df_e_23['system_ensemble_method_subsystem_count'].isna()]
df_24_noensemble = df_e_24.loc[df_e_24['system_ensemble_method_subsystem_count'].isna()]

#Ensemble
df_23_ensemble = df_e_23.loc[~df_e_23['system_ensemble_method_subsystem_count'].isna()]
df_24_ensemble = df_e_24.loc[~df_e_24['system_ensemble_method_subsystem_count'].isna()]

In [ ]:
print(len(df_23_noensemble), len(df_24_noensemble))

27 19


In [ ]:
print(len(df_23_ensemble), len(df_24_ensemble))

33 16


In [ ]:
df_baseline_23_1 = df_e_23[df_e_23['code'] == 'Baseline_task4a_1']
df_baseline_23_2 = df_e_23[df_e_23['code'] == 'Baseline_task4a_2']
df_23_ensemble_plus_baseline = pd.concat([df_23_ensemble, df_baseline_23_1,df_baseline_23_2])

df_baseline_24 = df_e_24[df_e_24['code'] == 'Baseline 2024']
df_24_ensemble_plus_baseline = pd.concat([df_24_ensemble, df_baseline_24])

In [ ]:
print_statistics([df_24_noensemble, df_24_ensemble], ['DCASE_24 Non Ensemble', 'DCASE_24 Ensemble'],['PSDS_1_all', 'PAUC_all', 'ranking_score'])


Statistics for df DCASE_24 Non Ensemble in terms of the metric considered
       PSDS_1_all PAUC_all ranking_score
count      19.000   19.000        19.000
mean        0.533    0.650         1.184
std         0.048    0.069         0.084
min         0.469    0.490         1.062
25%         0.491    0.607         1.124
median      0.528    0.667         1.189
75%         0.571    0.694         1.231
max         0.642    0.738         1.354


Statistics for df DCASE_24 Ensemble in terms of the metric considered
       PSDS_1_all PAUC_all ranking_score
count      16.000   16.000        16.000
mean        0.473    0.616         1.089
std         0.224    0.162         0.380
min         0.000    0.211         0.211
25%         0.497    0.613         1.169
median      0.551    0.690         1.215
75%         0.604    0.712         1.277
max         0.677    0.744         1.416




/tmp/ipykernel_10801/3221466737.py:12: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

/tmp/ipykernel_10801/3221466737.py:12: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



In [ ]:
print_statistics([df_23_noensemble, df_24_noensemble], ['DCASE23','DCASE_24'], ['macs', 'energy_training_normalized', 'energy_test_normalized'])

Statistics for df DCASE23 in terms of the metric considered
                    macs energy_training_normalized energy_test_normalized
count             27.000                     27.000                 27.000
mean     51109394925.926                      5.632                  0.118
std     114659805362.170                      6.642                  0.208
min        105200000.000                      0.192                  0.002
25%        911727000.000                      1.039                  0.010
median    3497000000.000                      3.241                  0.021
75%      11390500000.000                      8.194                  0.075
max     460000000000.000                     23.040                  0.640


Statistics for df DCASE_24 in terms of the metric considered
                   macs energy_training_normalized energy_test_normalized
count            19.000                     17.000                 17.000
mean    12076464208.842                      3.226    

/tmp/ipykernel_10801/3221466737.py:12: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

/tmp/ipykernel_10801/3221466737.py:12: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



In [ ]:
print_statistics([df_23_ensemble, df_24_ensemble], ['DCASE23','DCASE_24'], ['macs', 'energy_training_normalized', 'energy_test_normalized'])

Statistics for df DCASE23 in terms of the metric considered
                      macs energy_training_normalized energy_test_normalized
count               33.000                     33.000                 33.000
mean     1205320969696.970                     32.045                  0.515
std      4010393897202.583                     50.798                  1.138
min          350000000.000                      1.882                  0.010
25%         7300000000.000                      4.608                  0.038
median     88200000000.000                      9.707                  0.075
75%       476604000000.000                     20.457                  0.427
max     21036000000000.000                    190.549                  6.240


Statistics for df DCASE_24 in terms of the metric considered
                    macs energy_training_normalized energy_test_normalized
count             16.000                     13.000                 13.000
mean    250197757116.000          

/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')
/tmp/ipykernel_10801/3221466737.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted_statistics = statistics.applymap(lambda x: f'{x:.3f}')


## Energy and performance comparisons (2023 - 2024)

In [ ]:
# type scale axes
type_xaxes = 'linear'
type_yaxes = 'log'

# axes labels
x_axes_lab = 'PSDS'
y_axes_lab = 'Energy train norm. (kWh)'

# which column to consider for the dataframes
df_x = 'PSDS_1_all'
df_y = 'energy_training_normalized'

figure=1
title = f'plots/psds_energy_train_10_ensemble.pdf'
text = 'code'

# figure = figure + 1

# scatter_compare(df_e_22, df_e_23, df_x, df_y, x_axes_lab, y_axes_lab, text, title, x_axes_lab_2=x_axes_lab_2, type_xaxes=type_xaxes, type_yaxes=type_yaxes, add_base=True, comp_22_23=True)
scatter_bothyears(df_24_ensemble_plus_baseline, df_23_ensemble_plus_baseline, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, x_range=[0.46,0.7], y_range=[-0.6,2.5], top_10=True, save=True, legend=False)


In [ ]:
# type scale axes
type_xaxes = 'linear'
type_yaxes = 'log'

# axes labels
x_axes_lab = 'PSDS'
y_axes_lab = 'Energy train norm. (kWh)'

# which column to consider for the dataframes
df_x = 'PSDS_1_all'
df_y = 'energy_training_normalized'

figure=1
title = f'plots/psds_energy_train_10_noensemble.pdf'
text = 'code'

# figure = figure + 1

# scatter_compare(df_e_22, df_e_23, df_x, df_y, x_axes_lab, y_axes_lab, text, title, x_axes_lab_2=x_axes_lab_2, type_xaxes=type_xaxes, type_yaxes=type_yaxes, add_base=True, comp_22_23=True)
scatter_bothyears(df_24_noensemble, df_23_noensemble, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes,x_range=[0.46,0.7], y_range=[-0.6,2.5], top_10=True, save=True)


In [ ]:
# type scale axes
type_xaxes = 'linear'
type_yaxes = 'log'

# axes labels
x_axes_lab = 'PSDS'
y_axes_lab = 'Energy test norm. (kWh)'

# which column to consider for the dataframes
df_x = 'PSDS_1_all'
df_y = 'energy_test_normalized'

figure=1
title = f'plots/psds_energy_test_10_ensemble.pdf'
text = 'code'

# figure = figure + 1

# scatter_compare(df_e_22, df_e_23, df_x, df_y, x_axes_lab, y_axes_lab, text, title, x_axes_lab_2=x_axes_lab_2, type_xaxes=type_xaxes, type_yaxes=type_yaxes, add_base=True, comp_22_23=True)
scatter_bothyears(df_24_ensemble_plus_baseline, df_23_ensemble_plus_baseline, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, x_range=[0.42,0.7], y_range=[-2.5,0.6], top_10=True, save=True, legend=False)


In [ ]:
# type scale axes
type_xaxes = 'log'
type_yaxes = 'log'

# axes labels
x_axes_lab = 'MACS'
y_axes_lab = 'Energy test norm. (kWh)'

# which column to consider for the dataframes
df_x = 'macs'
df_y = 'energy_test_normalized'

figure=1
title = f'plots/macs_energy_test.pdf'
text = 'code'

# figure = figure + 1

# scatter_compare(df_e_22, df_e_23, df_x, df_y, x_axes_lab, y_axes_lab, text, title, x_axes_lab_2=x_axes_lab_2, type_xaxes=type_xaxes, type_yaxes=type_yaxes, add_base=True, comp_22_23=True)
scatter_bothyears(df_e_24, df_e_23, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, x_range=[6,14], save=True, legend=False)


In [ ]:
# type scale axes
type_xaxes = 'log'
type_yaxes = 'log'

# axes labels
x_axes_lab = 'System complexity (param)'
y_axes_lab = 'Energy test norm. (kWh)'

# which column to consider for the dataframes
df_x = 'system_complexity'
df_y = 'energy_test_normalized'

figure=1
title = f'plots/param_energy_test.pdf'
text = 'code'

# figure = figure + 1

# scatter_compare(df_e_22, df_e_23, df_x, df_y, x_axes_lab, y_axes_lab, text, title, x_axes_lab_2=x_axes_lab_2, type_xaxes=type_xaxes, type_yaxes=type_yaxes, add_base=True, comp_22_23=True)
scatter_bothyears(df_e_24, df_e_23, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, x_range=[4,10], save=True, legend=False)


## Energy-weighted performance (2024)

In [ ]:
# type scale axes
type_xaxes = 'linear'
type_yaxes = 'log'

# axes labels
x_axes_lab = 'PSDS'
y_axes_lab = 'EW-PSDS'

# which column to consider for the dataframes
df_x = 'PSDS_1_all'
df_y = 'ew_psds'

figure=1
title = f'plots/ew_psds_ensemble.pdf'
text = 'code'

x_range = [0, 0.75]
y_range = [0.002, 3]

scatter_2024_ensemble_four(df_24_ensemble, df_24_noensemble, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, x_range=x_range, y_range=y_range, save=True)

In [ ]:
# type scale axes
type_xaxes = 'linear'
type_yaxes = 'log'

# axes labels
x_axes_lab = 'segMPAUC'
y_axes_lab = 'EW-segMPAUC'

# which column to consider for the dataframes
df_x = 'PAUC_all'
df_y = 'ew_pauc'

figure=1
title = f'plots/ew_pauc_ensemble.pdf'
text = 'code'

x_range = [0, 0.82]
y_range = [0.002, 3]

scatter_2024_ensemble_four(df_24_ensemble, df_24_noensemble, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, x_range=x_range, y_range=y_range, save=True, legend=False)

## Threshold based on energy consumption (2023 - 2024)

In [ ]:
#2023 Ensemble
metrics = ['energy_training_normalized', 'macs', 'system_complexity']
quantiles = [0.25, 0.5, 0.75, 1]

for quantile in quantiles:
    print(f"Quantile considered: {quantile}")
    for metric in metrics:
        # Calculate the threshold value for 'energy_training_normalized' in the ensembled systems
        threshold_metric = df_23_ensemble[metric].quantile(quantile)

        # Filter and create a DataFrame 'df_under_threshold' containing systems with values below the median
        df_under_threshold = df_23_ensemble[df_23_ensemble[metric] < threshold_metric]
        df_under_threshold.sort_values(by=metric, ascending=False, inplace=True)

        # Print the calculated threshold value for 'energy_training_normalized'
        print(f"{metric} threshold: {threshold_metric:.2f}")

        df_under_threshold.sort_values(by='PSDS_1_all', ascending=False, inplace=True)
        print(df_under_threshold[:1]['PSDS_1_all'])
        print("\n")

    print("\n")


Quantile considered: 0.25
energy_training_normalized threshold: 4.61
56    0.575
Name: PSDS_1_all, dtype: float64


macs threshold: 7300000000.00
32    0.494
Name: PSDS_1_all, dtype: float64


system_complexity threshold: 16687376.00
57    0.61
Name: PSDS_1_all, dtype: float64




Quantile considered: 0.5
energy_training_normalized threshold: 9.71
56    0.575
Name: PSDS_1_all, dtype: float64


macs threshold: 88200000000.00
56    0.575
Name: PSDS_1_all, dtype: float64


system_complexity threshold: 64870560.00
58    0.61
Name: PSDS_1_all, dtype: float64




Quantile considered: 0.75
energy_training_normalized threshold: 20.46
58    0.61
Name: PSDS_1_all, dtype: float64


macs threshold: 476604000000.00
57    0.61
Name: PSDS_1_all, dtype: float64


system_complexity threshold: 452500000.00
34    0.621
Name: PSDS_1_all, dtype: float64




Quantile considered: 1
energy_training_normalized threshold: 190.55
34    0.621
Name: PSDS_1_all, dtype: float64


macs threshold: 21036000000000.00
34

/tmp/ipykernel_7966/755734005.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_7966/755734005.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_7966/755734005.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_7966/755734005.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.

In [ ]:
#2024 Non-Ensemble
metrics = ['energy_training_normalized', 'macs', 'system_complexity']
quantiles = [0.25, 0.5, 0.75,1]

for quantile in quantiles:
    print(f"Quantile considered: {quantile}")
    for metric in metrics:
        # Calculate the threshold value for 'energy_training_normalized' in the ensembled systems
        threshold_metric = df_24_noensemble[metric].quantile(quantile)

        # Filter and create a DataFrame 'df_under_threshold' containing systems with values below the median
        df_under_threshold = df_24_noensemble[df_24_noensemble[metric] < threshold_metric]
        df_under_threshold.sort_values(by=metric, ascending=False, inplace=True)

        # Print the calculated threshold value for 'energy_training_normalized'
        print(f"{metric} threshold: {threshold_metric:.2f}")

        df_under_threshold.sort_values(by='PSDS_1_all', ascending=False, inplace=True)
        print(df_under_threshold[:1][['code','PSDS_1_all','PAUC_all']])
        print("\n")

    print("\n")

Quantile considered: 0.25
energy_training_normalized threshold: 1.18
                 code  PSDS_1_all  PAUC_all
17  Chen_NCUT_task4_1        0.53     0.667


macs threshold: 1199000000.00
                       code  PSDS_1_all  PAUC_all
31  XIAO_FMSG-JLESS_task4_1       0.572      0.49


system_complexity threshold: 1590237.00
                  code  PSDS_1_all  PAUC_all
16  Huang_SJTU_task4_4       0.523     0.678




Quantile considered: 0.5
energy_training_normalized threshold: 1.99
                   code  PSDS_1_all  PAUC_all
2  Schmid_CPJKU_task4_2       0.642     0.711


macs threshold: 1792000000.00
                       code  PSDS_1_all  PAUC_all
27  XIAO_FMSG-JLESS_task4_2       0.593      0.53


system_complexity threshold: 3438938.00
                       code  PSDS_1_all  PAUC_all
27  XIAO_FMSG-JLESS_task4_2       0.593      0.53




Quantile considered: 0.75
energy_training_normalized threshold: 4.37
                   code  PSDS_1_all  PAUC_all
2  Schmid_CPJKU_task4_

/tmp/ipykernel_5913/1476384686.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5913/1476384686.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5913/1476384686.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5913/1476384686.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pyd

macs threshold: 45259000000.00
                   code  PSDS_1_all  PAUC_all
2  Schmid_CPJKU_task4_2       0.642     0.711


system_complexity threshold: 181600000.00
                   code  PSDS_1_all  PAUC_all
2  Schmid_CPJKU_task4_2       0.642     0.711






/tmp/ipykernel_5913/1476384686.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5913/1476384686.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5913/1476384686.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5913/1476384686.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pyd

In [ ]:
#2024 Ensemble
metrics = ['energy_training_normalized', 'macs', 'system_complexity']
quantiles = [0.25, 0.5, 0.75,1]

for quantile in quantiles:
    print(f"Quantile considered: {quantile}")
    for metric in metrics:
        # Calculate the threshold value for 'energy_training_normalized' in the ensembled systems
        threshold_metric = df_24_ensemble[metric].quantile(quantile)

        # Filter and create a DataFrame 'df_under_threshold' containing systems with values below the median
        df_under_threshold = df_24_ensemble[df_24_ensemble[metric] < threshold_metric]
        df_under_threshold.sort_values(by=metric, ascending=False, inplace=True)

        # Print the calculated threshold value for 'energy_training_normalized'
        print(f"{metric} threshold: {threshold_metric:.2f}")

        df_under_threshold.sort_values(by='PSDS_1_all', ascending=False, inplace=True)
        print(df_under_threshold[:1][['code','PSDS_1_all','PAUC_all']])
        print("\n")

    print("\n")

Quantile considered: 0.25
energy_training_normalized threshold: 5.92
                             code  PSDS_1_all  PAUC_all
13  Kim_GIST-HanwhaVision_task4_4       0.584     0.638


macs threshold: 15614000000.00
                       code  PSDS_1_all  PAUC_all
23  XIAO_FMSG-JLESS_task4_4       0.603     0.566


system_complexity threshold: 14993677.00
              code  PSDS_1_all  PAUC_all
18  LEE_KT_task4_4       0.506      0.69




Quantile considered: 0.5
energy_training_normalized threshold: 8.84
                             code  PSDS_1_all  PAUC_all
13  Kim_GIST-HanwhaVision_task4_4       0.584     0.638


macs threshold: 156714000000.00
                       code  PSDS_1_all  PAUC_all
23  XIAO_FMSG-JLESS_task4_4       0.603     0.566


system_complexity threshold: 257816736.00
                       code  PSDS_1_all  PAUC_all
23  XIAO_FMSG-JLESS_task4_4       0.603     0.566




Quantile considered: 0.75
energy_training_normalized threshold: 25.90
                       co

/tmp/ipykernel_10801/1336045584.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_10801/1336045584.py:18: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_10801/1336045584.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_10801/1336045584.py:18: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas

In [ ]:
#2023 Non-Ensemble
metrics = ['energy_training_normalized', 'macs', 'system_complexity']
quantiles = [0.25, 0.5, 0.75, 1]

for quantile in quantiles:
    print(f"Quantile considered: {quantile}")
    for metric in metrics:
        # Calculate the threshold value for 'energy_training_normalized' in the ensembled systems
        threshold_metric = df_23_noensemble[metric].quantile(quantile)

        # Filter and create a DataFrame 'df_under_threshold' containing systems with values below the median
        df_under_threshold = df_23_noensemble[df_23_noensemble[metric] < threshold_metric]
        df_under_threshold.sort_values(by=metric, ascending=False, inplace=True)

        # Print the calculated threshold value for 'energy_training_normalized'
        print(f"{metric} threshold: {threshold_metric:.2f}")

        df_under_threshold.sort_values(by='PSDS_1_all', ascending=False, inplace=True)
        print(df_under_threshold[:1][['code','PSDS_1_all']])
        print("\n")

    print("\n")


Quantile considered: 0.25
energy_training_normalized threshold: 1.00
                  code  PSDS_1_all
28  Xiao_FMSG_task4a_4        0.55


macs threshold: 911727000.00
                  code  PSDS_1_all
28  Xiao_FMSG_task4a_4        0.55


system_complexity threshold: 4460124.00
                  code  PSDS_1_all
28  Xiao_FMSG_task4a_4        0.55




Quantile considered: 0.5
energy_training_normalized threshold: 3.35
                 code  PSDS_1_all
22  Chen_CHT_task4a_2       0.561


macs threshold: 1824000000.00
                  code  PSDS_1_all
28  Xiao_FMSG_task4a_4        0.55


system_complexity threshold: 6600000.00
                              code  PSDS_1_all
69  Kim_GIST-HanwhaVision_task4a_2        0.59




Quantile considered: 0.75
energy_training_normalized threshold: 8.81
                              code  PSDS_1_all
69  Kim_GIST-HanwhaVision_task4a_2        0.59


macs threshold: 7300000000.00
                 code  PSDS_1_all
22  Chen_CHT_task4a_2       0.561


s

/tmp/ipykernel_5913/334678668.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5913/334678668.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5913/334678668.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_5913/334678668.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.

## BONUS : Train / Test GPU Consumption

In [ ]:
# type scale axes
type_xaxes = 'log'
type_yaxes = 'log'

# axes labels
x_axes_lab = 'GPU Energy train (kWh)'
y_axes_lab = 'Energy train (kWh)'

# which column to consider for the dataframes
df_x = 'gpu_energy_submission_training'
df_y = 'energy_training'

figure=1
title = f'plots/gpu_energy_train.pdf'
text = 'code'


scatter_2024(df_e_24, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, save=True)

In [ ]:
# type scale axes
type_xaxes = 'log'
type_yaxes = 'log'

# axes labels
x_axes_lab = 'GPU Energy train norm. (kWh)'
y_axes_lab = 'Energy train norm. (kWh)'

# which column to consider for the dataframes
df_x = 'gpu_energy_norm_training'
df_y = 'energy_training_normalized'

figure=1
title = f'plots/gpu_energy_norm_train.pdf'
text = 'code'

scatter_2024(df_e_24, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, save=True, legend=True)

In [ ]:
# type scale axes
type_xaxes = 'log'
type_yaxes = 'log'

# axes labels
x_axes_lab = 'GPU Energy test (kWh)'
y_axes_lab = 'Energy test (kWh)'

# which column to consider for the dataframes
df_x = 'gpu_energy_submission_test'
df_y = 'energy_test'

figure=1
title = f'plots/gpu_energy_test.pdf'
text = 'code'

scatter_2024(df_e_24, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, save=True)

In [ ]:
# type scale axes
type_xaxes = 'log'
type_yaxes = 'log'

# axes labels
x_axes_lab = 'GPU Energy test norm. (kWh)'
y_axes_lab = 'Energy test norm. (kWh)'

# which column to consider for the dataframes
df_x = 'gpu_energy_norm_test'
df_y = 'energy_test_normalized'

figure=1
title = f'plots/gpu_energy_norm_test.pdf'
text = 'code'

scatter_2024(df_e_24, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, save=True, legend=False)

In [ ]:
# type scale axes
type_xaxes = 'linear'
type_yaxes = 'linear'

# axes labels
x_axes_lab = '% GPU Train'
y_axes_lab = '% GPU Test'

# which column to consider for the dataframes
df_x = 'pourc_gpu_training'
df_y = 'pourc_gpu_test'

figure=1
title = f'plots/pourc_gpu_sumbission_train_test.pdf'
text = 'code'

scatter_2024(df_e_24, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, save=True)

In [ ]:
# type scale axes
type_xaxes = 'linear'
type_yaxes = 'linear'

# axes labels
x_axes_lab = '% GPU Train'
y_axes_lab = '% GPU Baseline'

# which column to consider for the dataframes
df_x = 'pourc_gpu_training'
df_y = 'pourc_gpu_training_baseline'

figure=1
title = f'plots/pourc_gpu_sumbission_baseline.pdf'
text = 'code'

scatter_2024(df_e_24, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, save=True)


In [ ]:
# type scale axes
type_xaxes = 'linear'
type_yaxes = 'linear'

# axes labels
x_axes_lab = '% GPU Train'
y_axes_lab = '% GPU Test'

# which column to consider for the dataframes
df_x = 'pourc_gpu_training'
df_y = 'pourc_gpu_test'

figure=1
title = f'plots/pourc_gpu_sumbission_baseline.pdf'
text = 'code'

scatter_2024(df_e_24, df_x, df_y, x_axes_lab, y_axes_lab, text, title, type_xaxes=type_xaxes, type_yaxes=type_yaxes, save=True)


## Check hardware used

In [ ]:
#Exporting hardware
df_focus_23 = df_e_23[['code','system_complexity_time', 'energy_baseline']]
df_focus_23['hardware'] = df_focus_23['system_complexity_time'].str.extract(r'\((.*?)\)')
df_focus_23['hardware'] = df_focus_23['hardware'].str.replace(r'^\d+\s*', '', regex=True)
df_focus_23['year']='2023'
df_focus_23.reset_index(inplace=True,drop=True)


df_focus_24 = df_e_24[['code','system_complexity_time', 'energy_baseline']].dropna()
df_focus_24['hardware'] = df_focus_24['system_complexity_time'].str.extract(r'\((.*?)\)')
df_focus_24['hardware'] = df_focus_24['hardware'].str.replace(r'^\d+\s*', '', regex=True)

df_focus_24['year']='2024'
df_focus_24.reset_index(inplace=True,drop=True)

df_focus_all = pd.concat([df_focus_23,df_focus_24])
df_focus_all=df_focus_all.sort_values(by='energy_baseline', ascending=False)
df_focus_all.to_csv(
    "sort_by_energy.csv",
    sep=';'
)
df_focus_all=df_focus_all.sort_values(by='hardware')
# df_focus_all.to_csv(
#     "sort_by_hardware.csv",
#     sep=';'
# )

/tmp/ipykernel_10801/2076803698.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_10801/2076803698.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_10801/2076803698.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

